In [1]:
from pathlib import Path
import sys

# Ensure project root is on path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Create output directory for DEC-002
output_dir = project_root / "outputs" / "dec002_baseline"
output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir)

Output directory: e:\slde-aft\outputs\dec002_baseline


In [2]:
import json

subset_file = project_root / "outputs" / "carb_10_dev" / "subset_10.jsonl"

with subset_file.open("r", encoding="utf-8") as f:
    subset_10 = [json.loads(line) for line in f if line.strip()]

print("Loaded", len(subset_10), "sentences")
print("Example sentence:", subset_10[0]["sentence"][:80], "...")

Loaded 10 sentences
Example sentence: 32.7 % of all households were made up of individuals and 15.7 % had someone livi ...


In [4]:
import os
from getpass import getpass
from openai import OpenAI

api_key = os.environ.get("OPENROUTER_API_KEY")

if not api_key:
    api_key = getpass("Enter your new OpenRouter API key: ").strip()
    os.environ["OPENROUTER_API_KEY"] = api_key

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    timeout=60.0,
    max_retries=1,
)

print("OpenRouter client initialized successfully.")

OpenRouter client initialized successfully.


In [6]:
from pathlib import Path
import sys
import json
import os
import time

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

output_dir = project_root / "outputs" / "dec002_baseline"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Output directory:", output_dir)

Project root: e:\slde-aft
Output directory: e:\slde-aft\outputs\dec002_baseline


In [7]:
subset_file = project_root / "outputs" / "carb_10_dev" / "subset_10.jsonl"

with subset_file.open("r", encoding="utf-8") as f:
    subset_10 = [json.loads(line) for line in f if line.strip()]

print("Loaded sentences:", len(subset_10))
print("First sentence:", subset_10[0]["sentence"])

Loaded sentences: 10
First sentence: 32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .


In [8]:
prompt_path = project_root / "prompts" / "openie_carb_v1.txt"

with prompt_path.open("r", encoding="utf-8") as f:
    base_prompt = f.read()

print("Prompt loaded from:", prompt_path)
print("Contains {SENTENCE} placeholder:", "{SENTENCE}" in base_prompt)

Prompt loaded from: e:\slde-aft\prompts\openie_carb_v1.txt
Contains {SENTENCE} placeholder: True


In [9]:
from getpass import getpass
from openai import OpenAI

api_key = os.environ.get("OPENROUTER_API_KEY")

if not api_key:
    api_key = getpass("Enter OpenRouter API key: ").strip()
    os.environ["OPENROUTER_API_KEY"] = api_key

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    timeout=60.0,
    max_retries=1,
)

BASELINE_MODEL = "meta-llama/llama-3.1-8b-instruct"
TEMPERATURE = 0.0

print("Client ready.")
print("Baseline model:", BASELINE_MODEL)

Client ready.
Baseline model: meta-llama/llama-3.1-8b-instruct


In [10]:
record = subset_10[0]
sentence = record["sentence"]
prompt = base_prompt.replace("{SENTENCE}", sentence)

print("Prompt contains actual sentence:", sentence in prompt)
print("Sending one test request...")

start = time.perf_counter()

try:
    response = client.chat.completions.create(
        model=BASELINE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
    )

    elapsed = time.perf_counter() - start
    raw_text = (response.choices[0].message.content or "").strip()

    print(f"Completed in {elapsed:.2f} seconds")
    print("Response:")
    print(raw_text)

except Exception as exc:
    elapsed = time.perf_counter() - start
    print(f"Failed after {elapsed:.2f} seconds")
    print(type(exc).__name__, str(exc))

Prompt contains actual sentence: True
Sending one test request...
Completed in 3.93 seconds
Response:
[
  {
    "subject": "all households",
    "predicate": "were made up of",
    "object": "individuals"
  },
  {
    "subject": "all households",
    "predicate": "had someone living alone",
    "object": "who was 65 years of age or older"
  }
]


In [11]:
baseline_predictions = []
latencies = []

for i, record in enumerate(subset_10, start=1):
    sentence = record["sentence"]
    gold_triples = record["gold_triples"]
    prompt = base_prompt.replace("{SENTENCE}", sentence)

    start = time.perf_counter()

    try:
        response = client.chat.completions.create(
            model=BASELINE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
        )

        raw_text = (response.choices[0].message.content or "").strip()

        parsed = json.loads(raw_text)

        if isinstance(parsed, dict) and "triples" in parsed:
            parsed_triples = parsed["triples"]
        elif isinstance(parsed, list):
            parsed_triples = parsed
        else:
            raise ValueError("Unexpected JSON response shape")

        error = None

    except Exception as exc:
        raw_text = ""
        parsed_triples = []
        error = f"{type(exc).__name__}: {exc}"

    elapsed = time.perf_counter() - start
    latencies.append(elapsed)

    baseline_predictions.append({
        "sentence_id": record.get("sentence_id", i),
        "sentence": sentence,
        "gold_triples": gold_triples,
        "predicted_triples": parsed_triples,
        "raw_response": raw_text,
        "latency_seconds": elapsed,
        "model": BASELINE_MODEL,
        "temperature": TEMPERATURE,
        "error": error,
    })

    print(
        f"Sentence {i:2d} | "
        f"latency: {elapsed:6.2f}s | "
        f"triples: {len(parsed_triples):2d} | "
        f"error: {error}"
    )

    time.sleep(1)

Sentence  1 | latency:   7.46s | triples:  2 | error: None
Sentence  2 | latency:   0.90s | triples:  1 | error: None
Sentence  3 | latency:   2.41s | triples:  3 | error: None
Sentence  4 | latency:   1.28s | triples:  3 | error: None
Sentence  5 | latency:   2.04s | triples:  6 | error: None
Sentence  6 | latency:   0.20s | triples:  3 | error: None
Sentence  7 | latency:   6.44s | triples:  3 | error: None
Sentence  8 | latency:  13.04s | triples:  4 | error: None
Sentence  9 | latency:   2.92s | triples:  1 | error: None
Sentence 10 | latency:   1.35s | triples:  3 | error: None


In [12]:
predictions_path = output_dir / "predictions.json"

with predictions_path.open("w", encoding="utf-8") as f:
    json.dump(baseline_predictions, f, ensure_ascii=False, indent=2)

config = {
    "experiment_id": "dec002_llama31_8b_carb10",
    "decision": "DEC-002",
    "dataset": "CaRB first 10 development sentences",
    "subset_file": str(subset_file),
    "model": BASELINE_MODEL,
    "provider": "OpenRouter",
    "prompt_file": str(prompt_path),
    "prompt_version": "openie_carb_v1",
    "temperature": TEMPERATURE,
    "api_calls": len(baseline_predictions),
    "average_latency_seconds": sum(latencies) / len(latencies) if latencies else None,
}

config_path = output_dir / "config.json"

with config_path.open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved predictions:", predictions_path)
print("Saved configuration:", config_path)
print("Average latency:", f"{config['average_latency_seconds']:.2f} seconds")

Saved predictions: e:\slde-aft\outputs\dec002_baseline\predictions.json
Saved configuration: e:\slde-aft\outputs\dec002_baseline\config.json
Average latency: 3.80 seconds


In [13]:
from src.evaluator import Triple, compute_precision_recall_f1

gold_triples = []
predicted_triples = []

for record in baseline_predictions:
    for triple in record["gold_triples"]:
        gold_triples.append(Triple(
            subject=triple["subject"],
            predicate=triple["predicate"],
            object=triple["object"],
        ))

    for triple in record["predicted_triples"]:
        predicted_triples.append(Triple(
            subject=triple["subject"],
            predicate=triple["predicate"],
            object=triple["object"],
        ))

baseline_metrics = compute_precision_recall_f1(
    gold_triples=gold_triples,
    predicted_triples=predicted_triples,
)

metrics_path = output_dir / "metrics_internal.json"

with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, ensure_ascii=False, indent=2)

print("DEC-002 baseline metrics")
for name, value in baseline_metrics.items():
    print(f"{name}: {value:.4f}" if isinstance(value, float) else f"{name}: {value}")

print("Saved metrics:", metrics_path)

DEC-002 baseline metrics
precision: 0.0345
recall: 0.0286
f1: 0.0312
true_positives: 1
false_positives: 28
false_negatives: 34
gold_count: 35
prediction_count: 29
Saved metrics: e:\slde-aft\outputs\dec002_baseline\metrics_internal.json


In [14]:
import json
from datetime import datetime

ours_metrics_path = project_root / "outputs" / "carb_10_dev" / "metrics_internal.json"
baseline_metrics_path = project_root / "outputs" / "dec002_baseline" / "metrics_internal.json"

with ours_metrics_path.open("r", encoding="utf-8") as f:
    ours_metrics = json.load(f)

with baseline_metrics_path.open("r", encoding="utf-8") as f:
    baseline_metrics = json.load(f)

comparison = {
    "experiment_id": "dec002_carb10_external_llm_comparison",
    "decision": "DEC-002",
    "dataset": "CaRB first 10 development sentences",
    "evaluation_protocol": (
        "Shared internal normalized exact-match triple evaluation. "
        "A match requires normalized subject, predicate, and object equality."
    ),
    "methods": [
        {
            "name": "SLDE-AFT CaRB pilot configuration",
            "model": "openrouter/auto",
            "prompt_version": "openie_carb_v1",
            "metrics": ours_metrics,
        },
        {
            "name": "External baseline: Llama-3.1-8B-Instruct",
            "model": "meta-llama/llama-3.1-8b-instruct",
            "prompt_version": "openie_carb_v1",
            "metrics": baseline_metrics,
        },
    ],
    "notes": [
        "This is a 10-sentence development pilot, not a full CaRB benchmark score.",
        "Results use strict normalized exact-match scoring rather than the official CaRB scorer.",
        "No KB context, feedback, synthetic data, or LoRA adapters were provided to the external baseline.",
    ],
}

comparison_path = output_dir / "comparison_carb10.json"

with comparison_path.open("w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

print("Saved comparison artifact:", comparison_path)

Saved comparison artifact: e:\slde-aft\outputs\dec002_baseline\comparison_carb10.json


In [15]:
import json
from src.evaluator import Triple, normalize_triple

baseline_path = project_root / "outputs" / "dec002_baseline" / "predictions.json"

with baseline_path.open("r", encoding="utf-8") as f:
    baseline_records = json.load(f)

examples = {
    "true_positive": [],
    "false_positive": [],
    "false_negative": [],
}

for record in baseline_records:
    gold_set = {normalize_triple(Triple(**triple)) for triple in record["gold_triples"]}
    predicted_set = {
        normalize_triple(Triple(**triple))
        for triple in record["predicted_triples"]
    }

    for triple in sorted(gold_set & predicted_set, key=lambda item: (item.subject, item.predicate, item.object)):
        examples["true_positive"].append({
            "sentence": record["sentence"],
            "triple": {
                "subject": triple.subject,
                "predicate": triple.predicate,
                "object": triple.object,
            },
        })

    for triple in sorted(predicted_set - gold_set, key=lambda item: (item.subject, item.predicate, item.object)):
        examples["false_positive"].append({
            "sentence": record["sentence"],
            "triple": {
                "subject": triple.subject,
                "predicate": triple.predicate,
                "object": triple.object,
            },
        })

    for triple in sorted(gold_set - predicted_set, key=lambda item: (item.subject, item.predicate, item.object)):
        examples["false_negative"].append({
            "sentence": record["sentence"],
            "triple": {
                "subject": triple.subject,
                "predicate": triple.predicate,
                "object": triple.object,
            },
        })

for category, values in examples.items():
    print(f"\n{category.upper().replace('_', ' ')}")
    print("-" * 70)

    for item in values[:3]:
        triple = item["triple"]
        print("Sentence:", item["sentence"])
        print("Triple:", f"({triple['subject']}; {triple['predicate']}; {triple['object']})")
        print()


TRUE POSITIVE
----------------------------------------------------------------------
Sentence: A partial list of turbomachinery that may use one or more centrifugal compressors within the machine are listed here .
Triple: (turbomachinery; may use; one or more centrifugal compressors)


FALSE POSITIVE
----------------------------------------------------------------------
Sentence: 32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .
Triple: (all households; had; someone living alone who was 65 years of age or older)

Sentence: 32.7 % of all households were made up of individuals and 15.7 % had someone living alone who was 65 years of age or older .
Triple: (all households; were made up of; individuals)

Sentence: A CEN forms an important but small part of a Local Strategic Partnership .
Triple: (a cen; forms; an important but small part of a local strategic partnership)


FALSE NEGATIVE
-----------------------------

In [16]:
import json

config_path = project_root / "outputs" / "dec002_baseline" / "config.json"

with config_path.open("r", encoding="utf-8") as f:
    baseline_config = json.load(f)

print("Average latency:", baseline_config.get("average_latency_seconds"))
print("API calls:", baseline_config.get("api_calls"))
print("Model:", baseline_config.get("model"))

Average latency: 3.804393349995371
API calls: 10
Model: meta-llama/llama-3.1-8b-instruct
